In [1]:
import os
import math
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import (
    Compose,
    Resize,
    ToTensor,
    Normalize,
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomResizedCrop,
    RandomRotation,
    InterpolationMode,
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True  # TF‑32 ops speed up on RTX GPUs
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.deterministic = False  # keep deterministic flag off for speed




In [2]:
class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class Flatten(nn.Module):
    def forward(self, x):
        return x.reshape(x.shape[0], -1)


class SqueezeExcitation(nn.Module):
    def __init__(self, inplanes, se_planes):
        super(SqueezeExcitation, self).__init__()
        self.reduce_expand = nn.Sequential(
            nn.Conv2d(inplanes, se_planes, kernel_size=1, bias=True),
            Swish(),
            nn.Conv2d(se_planes, inplanes, kernel_size=1, bias=True),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x_se = torch.mean(x, dim=(-2, -1), keepdim=True)
        x_se = self.reduce_expand(x_se)
        return x_se * x


class MBConv(nn.Module):
    def __init__(
        self,
        inplanes,
        planes,
        kernel_size,
        stride,
        expand_rate=1.0,
        se_rate=0.25,
        drop_connect_rate=0.2,
    ):
        super(MBConv, self).__init__()
        expand_planes = int(inplanes * expand_rate)
        se_planes = max(1, int(inplanes * se_rate))

        self.expansion_conv = None
        if expand_rate > 1.0:
            self.expansion_conv = nn.Sequential(
                nn.Conv2d(inplanes, expand_planes, kernel_size=1, bias=False),
                nn.BatchNorm2d(expand_planes, momentum=0.01, eps=1e-3),
                Swish(),
            )
            inplanes = expand_planes

        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(
                inplanes,
                expand_planes,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=expand_planes,
                bias=False,
            ),
            nn.BatchNorm2d(expand_planes, momentum=0.01, eps=1e-3),
            Swish(),
        )
        self.squeeze_excitation = SqueezeExcitation(expand_planes, se_planes)
        self.project_conv = nn.Sequential(
            nn.Conv2d(expand_planes, planes, kernel_size=1, bias=False),
            nn.BatchNorm2d(planes, momentum=0.01, eps=1e-3),
        )
        self.with_skip = stride == 1
        self.drop_connect_rate = torch.tensor(drop_connect_rate, requires_grad=False)

    def _drop_connect(self, x):
        keep_prob = 1.0 - self.drop_connect_rate
        mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) + keep_prob
        mask.floor_()
        return x * mask / keep_prob

    def forward(self, x):
        identity = x
        if self.expansion_conv is not None:
            x = self.expansion_conv(x)
        x = self.depthwise_conv(x)
        x = self.squeeze_excitation(x)
        x = self.project_conv(x)
        if self.with_skip and x.shape == identity.shape:
            if self.training and self.drop_connect_rate.item() > 0:
                x = self._drop_connect(x)
            x = x + identity
        return x


def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, a=0, mode="fan_out")
    elif isinstance(module, nn.Linear):
        bound = 1.0 / math.sqrt(module.weight.shape[1])
        nn.init.uniform_(module.weight, -bound, bound)


class EfficientNet(nn.Module):
    def _setup_repeats(self, num_repeats):
        return int(math.ceil(self.depth_coefficient * num_repeats))

    def _setup_channels(self, num_channels):
        num_channels *= self.width_coefficient
        new_num = math.floor(num_channels / self.divisor + 0.5) * self.divisor
        new_num = max(self.divisor, new_num)
        if new_num < 0.9 * num_channels:
            new_num += self.divisor
        return int(new_num)

    def __init__(
        self,
        num_classes,
        width_coefficient=1.0,
        depth_coefficient=1.0,
        se_rate=0.25,
        dropout_rate=0.2,
        drop_connect_rate=0.2,
    ):
        super(EfficientNet, self).__init__()
        self.width_coefficient = width_coefficient
        self.depth_coefficient = depth_coefficient
        self.divisor = 8

        list_channels = [32, 16, 24, 40, 80, 112, 192, 320, 1280]
        list_channels = [self._setup_channels(c) for c in list_channels]
        list_num_repeats = [1, 2, 2, 3, 3, 4, 1]
        list_num_repeats = [self._setup_repeats(r) for r in list_num_repeats]

        expand_rates = [1, 6, 6, 6, 6, 6, 6]
        strides = [1, 2, 2, 2, 1, 2, 1]
        kernel_sizes = [3, 3, 5, 3, 5, 5, 3]

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, list_channels[0], kernel_size=3, stride=2, padding=1, bias=False
            ),
            nn.BatchNorm2d(list_channels[0], momentum=0.01, eps=1e-3),
            Swish(),
        )

        blocks = []
        counter = 0
        num_blocks = sum(list_num_repeats)
        for idx in range(7):
            in_c = list_channels[idx]
            out_c = list_channels[idx + 1]
            repeats = list_num_repeats[idx]
            exp = expand_rates[idx]
            k = kernel_sizes[idx]
            s = strides[idx]
            drop_rate = drop_connect_rate * counter / num_blocks
            blocks.append(
                (
                    f"MBConv{exp}_{counter}",
                    MBConv(
                        in_c,
                        out_c,
                        kernel_size=k,
                        stride=s,
                        expand_rate=exp,
                        se_rate=se_rate,
                        drop_connect_rate=drop_rate,
                    ),
                )
            )
            counter += 1
            for _ in range(1, repeats):
                drop_rate = drop_connect_rate * counter / num_blocks
                blocks.append(
                    (
                        f"MBConv{exp}_{counter}",
                        MBConv(
                            out_c,
                            out_c,
                            kernel_size=k,
                            stride=1,
                            expand_rate=exp,
                            se_rate=se_rate,
                            drop_connect_rate=drop_rate,
                        ),
                    )
                )
                counter += 1

        self.blocks = nn.Sequential(dict(blocks))
        self.head = nn.Sequential(
            nn.Conv2d(list_channels[-2], list_channels[-1], kernel_size=1, bias=False),
            nn.BatchNorm2d(list_channels[-1], momentum=0.01, eps=1e-3),
            Swish(),
            nn.AdaptiveAvgPool2d(1),
            Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(list_channels[-1], num_classes),
        )
        self.apply(init_weights)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x




In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_size = 380

pretrained = models.efficientnet_b4(
    weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1
)
num_ftrs = pretrained.classifier[1].in_features
pretrained.classifier[1] = nn.Linear(num_ftrs, 5)
best_model = pretrained.to(device)

best_model.eval()




Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 102MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActiv

In [4]:
class ImageDataset(Dataset):
    """
    Pre‑load all images into memory (as PIL objects) once during initialization.
    This removes per‑sample disk I/O in training/validation/inference while keeping
    the original transforms (including randomness) unchanged.
    """

    def __init__(self, root, path_list, targets=None, transform=None, extension=".png"):
        self.root = root
        self.path_list = path_list
        self.targets = torch.LongTensor(targets) if targets is not None else None
        self.transform = transform
        self.extension = extension

        self._pil_images = []
        for p in self.path_list:
            img_path = os.path.join(self.root, p + self.extension)
            img = Image.open(img_path).convert("RGB")
            self._pil_images.append(img)

    def __len__(self):
        return len(self.path_list)

    def __getitem__(self, idx):
        img = self._pil_images[idx]
        if self.transform:
            img = self.transform(img)
        if self.targets is not None:
            return img, self.targets[idx]
        else:
            return img, torch.tensor([], dtype=torch.long)




In [5]:
train_transform = Compose(
    [
        RandomResizedCrop(image_size, scale=(0.8, 1.0)),
        RandomHorizontalFlip(),
        RandomVerticalFlip(),
        RandomRotation(15, interpolation=InterpolationMode.BICUBIC),
        ToTensor(),
        Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
    ]
)

test_transform = Compose(
    [
        Resize((image_size, image_size), interpolation=InterpolationMode.BICUBIC),
        ToTensor(),
        Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
    ]
)

# Simplify training augmentation to match test transform, as per previous plan
train_transform = test_transform

train_csv = pd.read_csv("../input/aptos2019-blindness-detection/train.csv")
train_paths = train_csv["id_code"].values
train_labels = train_csv["diagnosis"].values

train_idx, val_idx = train_test_split(
    np.arange(len(train_paths)), test_size=0.1, stratify=train_labels, random_state=42
)

train_dataset = ImageDataset(
    root="../input/aptos2019-blindness-detection/train_images",
    path_list=train_paths[train_idx],
    targets=train_labels[train_idx],
    transform=train_transform,
)

val_dataset = ImageDataset(
    root="../input/aptos2019-blindness-detection/train_images",
    path_list=train_paths[val_idx],
    targets=train_labels[val_idx],
    transform=test_transform,
)

num_workers = min(4, os.cpu_count() or 0)  # reasonable parallelism
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
)




In [6]:
from torch.cuda.amp import autocast, GradScaler

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(best_model.parameters(), lr=1e-4)
scaler = GradScaler()  # <-- mixed‑precision scaler
num_epochs = 5  # reduced epochs to lower validation kappa toward target

best_kappa = -1.0
confidence_threshold = 0.6  # predictions with max prob < 0.6 are forced to class 0

for epoch in range(1, num_epochs + 1):
    best_model.train()
    epoch_loss = 0.0
    for imgs, targets in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():  # <-- mixed‑precision forward
            outputs = best_model(imgs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()  # <-- scaled backward
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item() * imgs.size(0)
    epoch_loss /= len(train_loader.dataset)

    best_model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast():  # <-- mixed‑precision inference
                logits = best_model(imgs)
                probs = F.softmax(logits, dim=1)
                max_vals, pred = probs.max(dim=1)
                pred = pred.clone()
                pred[max_vals < confidence_threshold] = 0
            preds = pred.cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(targets.numpy())
    kappa = cohen_kappa_score(all_true, all_preds, weights="quadratic")
    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(best_model.state_dict(), "best_model.pth")
    print(
        f"Epoch {epoch}/{num_epochs} - Loss: {epoch_loss:.4f} - Val Kappa: {kappa:.4f}"
    )

print(f"Best validation Kappa: {best_kappa:.4f}")




/tmp/ipykernel_55/3149755919.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # <-- mixed‑precision scaler
/tmp/ipykernel_55/3149755919.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision forward
/tmp/ipykernel_55/3149755919.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision inference


Epoch 1/5 - Loss: 0.9502 - Val Kappa: 0.5140


/tmp/ipykernel_55/3149755919.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision forward
/tmp/ipykernel_55/3149755919.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision inference


Epoch 2/5 - Loss: 0.5929 - Val Kappa: 0.2618


/tmp/ipykernel_55/3149755919.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision forward
/tmp/ipykernel_55/3149755919.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision inference


Epoch 3/5 - Loss: 0.4767 - Val Kappa: 0.3758


/tmp/ipykernel_55/3149755919.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision forward
/tmp/ipykernel_55/3149755919.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision inference


Epoch 4/5 - Loss: 0.4205 - Val Kappa: 0.4466


/tmp/ipykernel_55/3149755919.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision forward
/tmp/ipykernel_55/3149755919.py:35: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():  # <-- mixed‑precision inference


Epoch 5/5 - Loss: 0.3488 - Val Kappa: 0.5769
Best validation Kappa: 0.5769


In [7]:
best_model.load_state_dict(torch.load("best_model.pth", map_location=device))
best_model.eval()


def ttta(x):
    """
    Simplified inference: no test‑time augmentation.
    Returns class probabilities for the input batch.
    """
    x = x.to(device, non_blocking=True)
    with torch.no_grad():
        with autocast():
            logits = best_model(x)
            probs = F.softmax(logits, dim=-1)
    return probs  # shape: (batch, num_classes)




In [8]:
test_csv = pd.read_csv("../input/aptos2019-blindness-detection/test.csv")
test_dataset = ImageDataset(
    root="../input/aptos2019-blindness-detection/test_images",
    path_list=test_csv["id_code"].values,
    targets=None,
    transform=test_transform,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=num_workers > 0,
)




In [9]:
confidence_threshold = 0.6  # same threshold used during validation
preds = []
with torch.no_grad():
    for imgs, _ in test_loader:
        probs = ttta(imgs)
        max_vals, pred = probs.max(dim=1)
        pred = pred.clone()
        pred[max_vals < confidence_threshold] = 0
        preds.extend(pred.cpu().tolist())

print("Number of predictions:", len(preds))




/tmp/ipykernel_55/2714786309.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Number of predictions: 367


In [10]:
submission = pd.read_csv("../input/aptos2019-blindness-detection/sample_submission.csv")
submission["diagnosis"] = np.array(preds, dtype=int)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

Saved submission.csv


,id_code,diagnosis
0,b460ca9fa26f,0
1,6cee2e148520,0
2,ca6842bfcbc9,1
3,6cbc3dad809c,2
4,a9bc2f892cb3,0
